In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# 1. Message based summarization
checkpointer = InMemorySaver()

agent = create_agent(
    model= model,
    tools=[],
    checkpointer=checkpointer,
    middleware=[
        SummarizationMiddleware(
            model= model,
             trigger=("messages", 6),        # Trigger summarization when message length reaches 10
            keep=("messages", 2)      # Keep the most recent 4 messages intact
        )
    ]
)

# Creating a configuration with a unique thread_id for memory tracking
config = {"configurable": {"thread_id": "summary_test_2"}}

questions = [
    "What is 2 + 2?",
    "What is 10 * 5?",
    "What is 100 / 4?",
    "what is good girl character ?",
    "what is 23*8 ?",
    "What is 15 - 7?"
]

# Testing the agent to see the message array compress after hitting 10 messages
for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config
    )

    print("\nQUESTION:", q)
    print("Returned messages:", len(response["messages"]))

    state = agent.get_state(config)

    print("Checkpoint messages:", len(state.values["messages"]))

    for i, msg in enumerate(state.values["messages"]):
        print(
            i,
            type(msg).__name__,
            "=>",
            str(msg.content)[:150]
        )

    print("=" * 60)


QUESTION: What is 2 + 2?
Returned messages: 2
Checkpoint messages: 2
0 HumanMessage => What is 2 + 2?
1 AIMessage => 2 + 2 = 4.

QUESTION: What is 10 * 5?
Returned messages: 4
Checkpoint messages: 4
0 HumanMessage => What is 2 + 2?
1 AIMessage => 2 + 2 = 4.
2 HumanMessage => What is 10 * 5?
3 AIMessage => 10 × 5 = 50.

QUESTION: What is 100 / 4?
Returned messages: 6
Checkpoint messages: 6
0 HumanMessage => What is 2 + 2?
1 AIMessage => 2 + 2 = 4.
2 HumanMessage => What is 10 * 5?
3 AIMessage => 10 × 5 = 50.
4 HumanMessage => What is 100 / 4?
5 AIMessage => 100 ÷ 4 = 25.

QUESTION: what is good girl character ?
Returned messages: 4
Checkpoint messages: 4
0 HumanMessage => Here is a summary of the conversation to date:

## SESSION INTENT
The user is requesting the result of a simple arithmetic calculation: 100 ÷ 4.

## S
1 AIMessage => 100 ÷ 4 = 25.
2 HumanMessage => what is good girl character ?
3 AIMessage => ### “Good Girl” Character – What It Means and Why It Matters

A **“good girl

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"Hotels in {city}:\n1. Grand Hotel - 5 star, $350/night, spa, pool, gym\n2. City Inn - 4 star, $180/night, business center\n3. Budget Stay - 3 star, $75/night, free wifi"

In [14]:
agent = create_agent(
    model= model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
           trigger=("tokens", 550),
           keep=("tokens", 200)
        )
    ]
)

In [15]:
def count_tokens(messages):
    total_chars = sum(len(m.content) for m in messages if hasattr(m, "content") and isinstance(m.content, str))
    return total_chars // 4

In [18]:
config = {"configurable": {"thread_id": "token_test_1"}}

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:

    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    print(f"\n{'=' * 70}")
    print(f"CITY: {city}")
    print(f"MESSAGES: {len(response['messages'])}")

    for i, msg in enumerate(response["messages"]):
        print(f"\n{i} - {type(msg).__name__}")
        print(str(msg.content)[:500])


CITY: Paris
MESSAGES: 7

0 - HumanMessage
Here is a summary of the conversation to date:

## SESSION INTENT  
The user wants to find hotels in Dubai.

## SUMMARY  
The user has requested hotels in Dubai. No search has been performed yet. No artifacts have been created or modified.

## ARTIFACTS  
None.

## NEXT STEPS  
Invoke `search_hotels` with `city: "Dubai"` to retrieve hotel options.

1 - HumanMessage
Find hotels in Paris

2 - AIMessage


3 - ToolMessage
Hotels in Dubai:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $180/night, business center
3. Budget Stay - 3 star, $75/night, free wifi

4 - AIMessage


5 - ToolMessage
Hotels in Paris:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $180/night, business center
3. Budget Stay - 3 star, $75/night, free wifi

6 - AIMessage
Here are some hotel options in Paris (based on the latest search results):

| # | Hotel Name   | Star Rating | Price per Night | Key Amenities               